* 최신 vllm 버전 요구됨, 리눅스 24.04 이상 권장

In [1]:
from vllm import LLM, SamplingParams
from vllm.sampling_params import StructuredOutputsParams
from transformers import AutoTokenizer
from datasets import Dataset

import pandas as pd
import numpy as np

import torch
import json

/root/anaconda3/envs/vLLM/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
def build_system_content(original_text):
    return (
        "You are an expert medical evaluator.\n\n"
        "Below is the ORIGINAL discharge summary of a patient.\n"
        "This is the ground-truth clinical record.\n\n"
        "===== ORIGINAL DISCHARGE SUMMARY =====\n"
        f"{original_text}\n"
        "======================================\n\n"
        "You will be given 5 generated clinical summaries of the SAME patient.\n"
        "Evaluate each summary independently, but assign scores based on RELATIVE quality,\n"
        "following the priority order specified below.\n\n"
        "EVALUATION PRIORITIES (VERY IMPORTANT):\n\n"
        "Priority 1 (MOST IMPORTANT): Correct use of REQUIRED TEST NAMES and SEX\n"
        "- The following test names must be explicitly and correctly written:\n"
        "  - Vitals: BP, HR, RR, Temp\n"
        "  - Labs: WBC, RBC, Hgb, Hct, Plt\n"
        "  - Red cell indices and others: MCV, MCH, MCHC, RDW, Glucose\n"
        "- Patient sex (Male/Female) must be correctly stated\n"
        "- Incorrect, missing, or improperly named tests or sex are penalized\n"
        "- This priority ALWAYS outweighs all lower priorities\n\n"
        "Priority 2: Correctness of VALUES corresponding to each test name\n"
        "- When a value is explicitly stated, it must match the discharge summary exactly\n"
        "- If a test is NOT present in the discharge summary, NA / not available / not reported\n"
        "  is considered CORRECT\n"
        "- Hallucinated, incorrect, or contradictory values are penalized\n\n"
        "Priority 3: Appropriateness of the clinical finding sentence\n"
        "- The sentence following \"The most diagnostic relevant finding was ...\" should:\n"
        "  - Describe a clinically meaningful finding or observation\n"
        "  - NOT be merely a diagnosis name\n"
        "  - Be consistent with the discharge summary\n\n"
        "SCORING RULES:\n"
        "- Scores represent RELATIVE ranks among the 5 summaries (5 = worst, 1 = best)\n"
        "- Multiple summaries may share the same score if their quality is comparable\n"
        "- Even if all summaries are poor, you MUST still rank them relative to each other\n\n"
        "OUTPUT RULES (STRICT):\n"
        "- You MUST output ONLY a single-line valid JSON object\n"
        "- The JSON must map summary index to score\n"
        "- Any token outside the JSON object is STRICTLY FORBIDDEN\n"
        "- Do NOT provide explanations, reasoning, comments, or natural language\n"
        "- Do NOT use markdown or headings\n\n"
        "Example:\n"
        "{\"1\": 1, \"2\": 3, \"3\": 2, \"4\": 4, \"5\": 5}"
    )

In [3]:
model_name = "Qwen/Qwen3-30B-A3B"

In [4]:
llm = LLM(
    model = model_name,
    dtype = torch.bfloat16,
    trust_remote_code = True,
    max_model_len = 32768,
    gpu_memory_utilization = 0.9
)

INFO 04-09 04:44:35 [utils.py:233] non-default args: {'trust_remote_code': True, 'dtype': torch.bfloat16, 'max_model_len': 32768, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-30B-A3B'}
INFO 04-09 04:44:37 [model.py:549] Resolved architecture: Qwen3MoeForCausalLM
INFO 04-09 04:44:37 [model.py:1678] Using max model len 32768


2026-04-09 04:44:37,767	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


INFO 04-09 04:44:37 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=16384.
INFO 04-09 04:44:37 [vllm.py:790] Asynchronous scheduling is enabled.
(EngineCore pid=234159) INFO 04-09 04:44:40 [core.py:105] Initializing a V1 LLM engine (v0.19.0) with config: model='Qwen/Qwen3-30B-A3B', speculative_config=None, tokenizer='Qwen/Qwen3-30B-A3B', skip_tokenizer_init=False, tokenizer_mode=auto, revision=None, tokenizer_revision=None, trust_remote_code=True, dtype=torch.bfloat16, max_seq_len=32768, download_dir=None, load_format=auto, tensor_parallel_size=1, pipeline_parallel_size=1, data_parallel_size=1, decode_context_parallel_size=1, dcp_comm_backend=ag_rs, disable_custom_all_reduce=False, quantization=None, enforce_eager=False, enable_return_routed_experts=False, kv_cache_dtype=auto, device_config=cuda, structured_outputs_config=StructuredOutputsConfig(backend='auto', disable_any_whitespace=False, disable_additional_properties=False, reasoning_parser='', reasoning_pa

(EngineCore pid=234159) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
(EngineCore pid=234159) <frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.nvrtc module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.nvrtc module instead.
Loading safetensors checkpoint shards:   0% Completed | 0/16 [00:00<?, ?it/s]
Loading safetensors checkpoint shards:   6% Completed | 1/16 [00:00<00:07,  2.12it/s]
Loading safetensors checkpoint shards:  12% Completed | 2/16 [00:00<00:06,  2.05it/s]
Loading safetensors checkpoint shards:  19% Completed | 3/16 [00:01<00:06,  2.04it/s]
Loading safetensors checkpoint shards:  25% Completed | 4/16 [00:01<00:06,  1.98it/s]
Loading safetensors checkpoint shards:  31% Completed | 5/16 [00:02<00:05,  1.96it/s]
Loading safetensors checkpoint shards:  38% C

(EngineCore pid=234159) INFO 04-09 04:44:52 [default_loader.py:384] Loading weights took 8.15 seconds
(EngineCore pid=234159) INFO 04-09 04:44:52 [gpu_model_runner.py:4820] Model loading took 56.88 GiB memory and 9.702730 seconds
(EngineCore pid=234159) INFO 04-09 04:44:55 [backends.py:1051] Using cache directory: /root/.cache/vllm/torch_compile_cache/12723b14e8/rank_0_0/backbone for vLLM's torch.compile
(EngineCore pid=234159) INFO 04-09 04:44:55 [backends.py:1111] Dynamo bytecode transform time: 2.80 s
(EngineCore pid=234159) INFO 04-09 04:44:58 [backends.py:285] Directly load the compiled graph(s) for compile range (1, 16384) from the cache, took 1.989 s
(EngineCore pid=234159) INFO 04-09 04:44:58 [decorators.py:303] Directly load AOT compilation from path /root/.cache/vllm/torch_compile_cache/torch_aot_compile/a1c4837f13bb34e5d638cb09461225233da38393b0e3b810fbfcc829c4172126/rank_0_0/model
(EngineCore pid=234159) INFO 04-09 04:44:58 [monitor.py:48] torch.compile took 5.12 s in total

(EngineCore pid=234159) 2026-04-09 04:45:00,548 - INFO - autotuner.py:262 - flashinfer.jit: [Autotuner]: Autotuning process starts ...
(EngineCore pid=234159) 2026-04-09 04:45:00,577 - INFO - autotuner.py:268 - flashinfer.jit: [Autotuner]: Autotuning process ends
Capturing CUDA graphs (mixed prefill-decode, PIECEWISE): 100%|██████████| 51/51 [00:03<00:00, 14.04it/s]
Capturing CUDA graphs (decode, FULL): 100%|██████████| 51/51 [00:02<00:00, 17.27it/s]


(EngineCore pid=234159) INFO 04-09 04:45:07 [gpu_model_runner.py:6046] Graph capturing finished in 7 secs, took 1.41 GiB
(EngineCore pid=234159) INFO 04-09 04:45:07 [gpu_worker.py:597] CUDA graph pool memory: 1.41 GiB (actual), 1.24 GiB (estimated), difference: 0.18 GiB (12.6%).
(EngineCore pid=234159) INFO 04-09 04:45:07 [core.py:283] init engine (profile, create kv cache, warmup model) took 15.20 seconds


(EngineCore pid=234159) INFO 04-09 04:57:10 [core.py:1210] Shutdown initiated (timeout=0)
(EngineCore pid=234159) INFO 04-09 04:57:10 [core.py:1233] Shutdown complete


In [5]:
df_gen = pd.read_csv("data/generated_data_v1.1.2.csv")
df_discharge = pd.read_csv("data/gen_data_for_dpo_20260221.csv")

In [6]:
summarized_text = df_gen.pivot_table(index = "subject_id", values = "generated_text",
                                     aggfunc = (lambda x: "\n\n".join([f"[{i+1}{"st" if i == 0 else "nd" if i == 1 else "rd" if i == 2 else "th"} generated text]\n\"\"\"\n{t}\n\"\"\"" for i, t in enumerate(x)])))\
                                        .reset_index()
df_gen = df_gen.assign(gen_num = [(i%5)+1 for i in range(df_gen.shape[0])])
df_wide = df_gen.pivot(index = "subject_id", columns = "gen_num", values = "generated_text").reset_index()
full_text = pd.merge(summarized_text, df_discharge[["subject_id", "text"]]).assign(system = lambda _df: _df.text.map(build_system_content)).drop(["text"], axis = 1)

In [7]:
ds = Dataset.from_pandas(full_text)
columns_to_remove = [f for f in list(ds.features) if f not in "subject_id"]

In [8]:
ds = ds.map(
    lambda sample:
    {"messages": [
        {"role": "system", "content": sample["system"]},
        {"role": "user", "content": sample["generated_text"]}
    ]}
)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 12943.88 examples/s]


In [9]:
ds = ds.map(remove_columns = columns_to_remove, batched = False)

Map: 100%|██████████| 2000/2000 [00:00<00:00, 37869.43 examples/s]


In [10]:
sampling_params = SamplingParams(temperature = 0.0, max_tokens = 64, structured_outputs=StructuredOutputsParams(json={"type": "object"}))
tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast = True)

def template_dataset(example):
    return {"prompt": tokenizer.apply_chat_template(example["messages"], tokenize = False, add_generation_prompt = True)}

inference_data = ds.map(template_dataset, remove_columns = ["messages"])
prompts = inference_data["prompt"]

Map: 100%|██████████| 2000/2000 [00:00<00:00, 10728.55 examples/s]


In [11]:
outputs = llm.generate(prompts, sampling_params)

data = []
idx = inference_data["subject_id"]

for i, output in enumerate(outputs):
    current_subject_id = idx[i]

    try:
        label = json.loads(output.outputs[0].text.strip())
    except:
        label = pd.NA

    row = {
        "subject_id": current_subject_id,
        "label": label
    }
    
    data.append(row)

df = pd.DataFrame(data)

Processed prompts: 100%|██████████| 2000/2000 [02:31<00:00, 13.24it/s, est. speed input: 55790.54 toks/s, output: 481.66 toks/s] 


In [13]:
df

,subject_id,label
0,10005808,"{'1': 5, '2': 4, '3': 2, '4': 3, '5': 1}"
1,10006820,"{'1': 5, '2': 3, '3': 4, '4': 2, '5': 1}"
2,10011668,"{'1': 2, '2': 3, '3': 4, '4': 1, '5': 5}"
3,10011912,"{'1': 5, '2': 5, '3': 3, '4': 4, '5': 1}"
4,10013700,"{'1': 5, '2': 1, '3': 5, '4': 2, '5': 3}"
...,...,...
1995,19973617,"{'1': 5, '2': 1, '3': 3, '4': 4, '5': 2}"
1996,19984875,"{'1': 3, '2': 4, '3': 2, '4': 5, '5': 1}"
1997,19987389,"{'1': 5, '2': 5, '3': 4, '4': 5, '5': 5}"
1998,19989918,"{'1': 5, '2': 2, '3': 4, '4': 5, '5': 1}"


In [43]:
data = []

for id, label in df.itertuples(index = False):
    if set(label.keys()) == {"1", "2", "3", "4", "5"} and max(label.values()) == 5 and min(label.values()) == 1:
        ## 순위의 중복이 있을 경우 제일 먼저 것만 선택
        max_idx = np.argmax(list(label)) + 1
        min_idx = np.argmin(list(label)) + 1
    else:
        continue

    row = {
        "subject_id": id,
        "text": df_discharge.loc[df_discharge.subject_id == id, "text"].item(),
        "chosen": df_wide.loc[df_wide.subject_id == id, max_idx].item(),
        "rejected": df_wide.loc[df_wide.subject_id == id, min_idx].item()
    }

    data.append(row)

dpo_dataset = pd.DataFrame(data)
# dpo_dataset.to_csv("data/dpo_dataset.csv", index = False, encoding = "utf-8-sig")

In [44]:
dpo_dataset.shape

(1814, 4)